# Fine-tune a Family Request Router (Qwen3)

**Week 5 project — custom-dataset variant.** Run this on a **free T4 GPU runtime** — both
Google Colab and Kaggle Notebooks work (Section 1 auto-detects which one you're on). On
Kaggle, enable Settings > Accelerator > GPU and Settings > Internet > On before running.

This follows the same recipe as the reference project (fine-tune `Qwen/Qwen3-1.7B-Base`
with a LoRA adapter through the [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)
visual UI), applied to a domain I actually own: routing requests inside my
[family-calendar](https://github.com/anushaakkiraju26/family-calendar) Deep Agent project
instead of IT support tickets.

Dataset: `data/family_request_routing.csv` in this repo — 30 real, eval-labelled parent
requests from family-calendar's evaluation suite, expanded with synthetic and hand-written
examples via `tools/generate_dataset.py` to ~50-70 balanced examples per label. See that
script and `data/family_request_routing_manifest.json` for exact provenance.

---
## The scenario

The Family Coordinator in my project is a Deep Agent: every incoming parent request first
gets reasoned about by a large hosted model to decide *how* to handle it — a single direct
tool call, an outright rejection, a scheduling-conflict resolution, a full multi-agent
weekly-planning workflow, a clarifying question, or an outing-research call. That routing
decision happens on **every** request, before any real work starts, and today it costs a full
frontier-model call each time.

```
fast_path_mutate      -> direct create/update/move/delete/restore, pending approval
fast_path_read        -> direct list/show, no mutation
fast_path_reject      -> deterministically declined (past event, cross-family, stale plan, ...)
fast_path_conflict    -> same child/parent double-booked; offers alternatives, may flag parent error
deep_weekly_workflow  -> full specialist pipeline: intake, planner, transportation, review, reminders
outing_workflow       -> Family Outing Agent's constrained search wrapper
ambiguous_clarify     -> missing info, must ask before acting
```

The model never plans the week or drafts a reminder itself — it just predicts which of these
seven paths a request belongs to, the same way the reference project's router predicts a
support-ticket queue. The rest of the coordinator's existing tools and specialists take over
from there.

`fast_path_conflict` was split out from `fast_path_reject` after an early fine-tuning pass:
a scheduling conflict (the same child double-booked) is a genuinely different downstream
action from a deterministic decline — the coordinator can offer alternative times or flag a
likely parent error, rather than rejecting outright — so it needed its own routing label
rather than being folded into `fast_path_reject`.

**Why not just call the frontier model for this every time?** That's the actual production
setup right now, and it works — but a routing decision on a short message does not need a
frontier model's full reasoning. A small fine-tuned classifier can make the same call in
milliseconds on modest hardware, and only escalate to the full agent once routing is already
decided.

## 1. Install dependencies

In [ ]:
import os
from pathlib import Path

# Portable working directory: Colab uses /content, Kaggle uses /kaggle/working
# (and needs Settings -> Internet: On and Settings -> Accelerator: GPU set
# explicitly). Every path below is built from WORKDIR instead of a hardcoded
# platform-specific path, so the rest of this notebook runs unchanged on
# either platform.
if Path("/content").exists():
    WORKDIR = "/content"
elif Path("/kaggle/working").exists():
    WORKDIR = "/kaggle/working"
else:
    WORKDIR = str(Path.cwd())

os.environ["WORKDIR"] = WORKDIR
print(f"Detected environment -> WORKDIR = {WORKDIR}")

In [ ]:
%cd $WORKDIR
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print(
      "Please set up a GPU before using LLaMA Factory: on Colab, "
      "Runtime > Change runtime type > T4 GPU; on Kaggle, "
      "Settings > Accelerator > GPU (and Settings > Internet > On)."
  )

## 2. Prepare the family-request routing dataset (INPUT REQUIRED)

The labelled CSV already lives in this repo, so the simplest path is to clone it straight
into this runtime. If the repo isn't reachable (private, not pushed yet, no internet access
enabled, etc.), this falls back to a manual upload widget for `family_request_routing.csv`
on Colab — on Kaggle, use "Add Input" to attach the CSV instead and set `CSV_PATH` manually.

This cell filters to the seven known labels, creates a **stratified 80/20 train/val split**,
converts the training split to **ShareGPT JSON**, writes it to `TRAIN.json` in the
LLaMA-Factory data dir, and registers it in `dataset_info.json` under the name
`family_request_routing` so it shows up in the LLaMA Board UI.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# ── constants ────────────────────────────────────────────────────────────────
# WORKDIR comes from the environment-detection cell in Section 1.
LLAMA_DATA_DIR  = f"{WORKDIR}/LLaMA-Factory/data"
TRAIN_JSON_PATH = f"{LLAMA_DATA_DIR}/TRAIN.json"
DATASET_INFO    = f"{LLAMA_DATA_DIR}/dataset_info.json"
REPO_URL        = "https://github.com/anushaakkiraju26/family-request-router.git"
REPO_DIR        = f"{WORKDIR}/family-request-router"

LABEL2ID = {
    "fast_path_mutate":     0,
    "fast_path_read":       1,
    "fast_path_reject":     2,
    "fast_path_conflict":   3,
    "deep_weekly_workflow": 4,
    "outing_workflow":      5,
    "ambiguous_clarify":    6,
}

# Single source of truth for label strings + their human-readable names —
# every later cell (baseline eval, classify(), confusion matrix, charts)
# reads LABEL_TOKENS / LABEL_DISPLAY / display_labels from here rather than
# redeclaring them, so there's one place to edit if labels ever change.
LABEL_TOKENS = list(LABEL2ID)
LABEL_DISPLAY = {
    "fast_path_mutate":     "Schedule change",
    "fast_path_read":       "List / show",
    "fast_path_reject":     "Declined",
    "fast_path_conflict":   "Conflict found",
    "deep_weekly_workflow": "Weekly plan",
    "outing_workflow":      "Outing search",
    "ambiguous_clarify":    "Needs clarification",
}
display_labels = [LABEL_DISPLAY[l] for l in LABEL_TOKENS]

# v3 system prompt — 7 labels now (fast_path_conflict split out of
# fast_path_reject: same-child/parent double-booking is a genuinely
# different downstream action than an outright decline). Same discipline as
# v2: explicit disambiguating detail per label, with reject and conflict now
# each explicitly saying what they are NOT to keep the split clean. This
# exact string must also be used at inference time in cell 16's classify().
SYSTEM_PROMPT = (
    "You are a family-calendar coordinator's routing assistant. Given a parent's "
    "request, respond with exactly one of the following seven categories:\n"
    "- fast_path_mutate: a single, specific create/update/move/delete/restore of "
    "one event or reminder, pending approval. Use this for ordinary cancel/change/"
    "add requests with no past-dated event, no scheduling conflict, and no "
    "cross-family issue.\n"
    "- fast_path_read: a single, specific read-only list/show request for events "
    "already on the calendar. No mutation.\n"
    "- fast_path_reject: use ONLY when the request itself states an explicit "
    "disqualifying fact that is NOT a scheduling conflict — a clearly past-dated "
    "event, cross-family access (the named child/parent does not belong to the "
    "stated family), a stale plan version, or a repeat of something already "
    "rejected. Do not use this as a default when merely uncertain.\n"
    "- fast_path_conflict: use when the request would double-book the SAME child "
    "or parent into two overlapping activities or pickups. This is different from "
    "fast_path_reject — a conflict can be offered alternatives, not just declined.\n"
    "- deep_weekly_workflow: a broad request spanning multiple days or a whole "
    "week, not one event.\n"
    "- outing_workflow: a request to search for or suggest NEW outing/activity "
    "ideas — not to list or show events already on the calendar.\n"
    "- ambiguous_clarify: use ONLY when a specific required detail is missing "
    "(which event, which family, which time) and no other label clearly fits. Do "
    "not use this as a default when merely uncertain either.\n"
    "Respond with exactly one label and nothing else."
)

# ── 1. Get the CSV: clone this repo, or fall back to manual upload ────────────
csv_path = Path(REPO_DIR) / "data" / "family_request_routing.csv"
if not csv_path.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=False)

if csv_path.exists():
    CSV_PATH = str(csv_path)
    print(f"Using cloned dataset: {CSV_PATH}")
else:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(
            "Could not clone the dataset repo and no Colab upload widget is "
            "available in this environment. On Kaggle: use 'Add Input' to "
            "attach family_request_routing.csv, then set CSV_PATH to its path "
            "(usually /kaggle/input/<dataset-name>/family_request_routing.csv) "
            "and re-run from the next cell."
        )
    print("Repo clone unavailable — upload family_request_routing.csv manually:")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

# ── 2. Load + filter ────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH).rename(columns={"category_truth": "label"})
df = df[df["label"].isin(LABEL2ID)].sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Loaded {len(df):,} rows")
print(df["label"].value_counts())

# ── 3. Stratified train/val split ───────────────────────────────────────────
df_train, df_val = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42,
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
print(f"\nTrain: {len(df_train):,} rows | Val (held-out): {len(df_val):,} rows")
print("\nTrain label distribution:")
print(df_train["label"].value_counts())

# ── 4. Convert train split -> ShareGPT JSON ─────────────────────────────────
sharegpt_records = [
    {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Parent request: {row['text']}"},
            {"role": "assistant", "content": row["label"]},
        ]
    }
    for _, row in df_train.iterrows()
]

with open(TRAIN_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(sharegpt_records, f, indent=2, ensure_ascii=False)
print(f"\nWritten {len(sharegpt_records):,} records -> {TRAIN_JSON_PATH}")

# ── 5. Register dataset in dataset_info.json ────────────────────────────────
with open(DATASET_INFO, "r", encoding="utf-8") as f:
    info = json.load(f)

info["family_request_routing"] = {
    "file_name": "TRAIN.json",
    "formatting": "sharegpt",
    "columns":   {"messages": "messages"},
    "tags": {
        "role_tag":      "role",
        "content_tag":   "content",
        "user_tag":      "user",
        "assistant_tag": "assistant",
        "system_tag":    "system",
    },
}

with open(DATASET_INFO, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
print(f"Registered 'family_request_routing' in {DATASET_INFO}")

# ── 6. Save val split for evaluation ────────────────────────────────────────
VAL_CSV = f"{WORKDIR}/val_split.csv"
df_val.to_csv(VAL_CSV, index=False)
print(f"Val split saved -> {VAL_CSV}  ({len(df_val):,} rows)")

## 3. Fine-tune via LLaMA Board (INPUT REQUIRED)

1. Run the next cell to start the LLaMA Board server, then open the **public** URL it prints.
2. Base model: `Qwen/Qwen3-1.7B-Base`. Dataset: `family_request_routing`. Finetuning: **LoRA**.
   Defaults are fine for a first run.
3. When training finishes, note the **Output Dir** path (`train_2026-...`) — you'll need it below.
4. **Manually stop this cell** once training completes — neither Colab nor Kaggle will stop
   the server for you.

Watch for the "training completed" message in the UI or the logs. Occasional "syntax error"
toasts from LLaMA Board are a known cosmetic issue and don't affect training. A short LoRA run
on a few hundred training rows usually takes well under the reference project's 30–60 min
estimate.

### Hyperparameters, briefly

Defaults are fine for a first pass. If you need to adjust:

| Knob | Effect | If your run is off |
|---|---|---|
| Learning rate | step size per update | loss oscillating -> lower it; loss barely moving -> raise it carefully |
| Epochs | passes over training data | loss still falling at the end -> add an epoch; val score drops late -> stop earlier |
| Batch size | examples per gradient step | tune for GPU memory / gradient noise |
| LoRA rank | adapter capacity | raise only if the task clearly needs more expressiveness — most of these seven labels don't, but the reject/conflict/clarify boundary might |

In [ ]:
%cd $WORKDIR/LLaMA-Factory
!GRADIO_SHARE=1 llamafactory-cli webui

---
## 4. Review training — loss curve (INPUT REQUIRED)

Set `ADAPTER_DIR` to the Output Dir path from LLaMA Board. The loss should drop and level
off; flat or chaotic usually means the learning rate, dataset size, or chat template is off.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

ADAPTER_DIR = f"{WORKDIR}/LLaMA-Factory/saves/Qwen3-1.7B-Base/lora/train_XXXX-XX-XX-XX-XX-XX"  # <-- CHANGE THIS
MERGED_DIR      = f"{WORKDIR}/family_router_merged"
BASE_MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

log_file = Path(ADAPTER_DIR) / "trainer_log.jsonl"
if not log_file.exists():
    raise FileNotFoundError(f"trainer_log.jsonl not found in {ADAPTER_DIR!r}")

records = [json.loads(l) for l in log_file.read_text().splitlines() if l.strip()]
steps   = [r["current_steps"] for r in records if r.get("loss") is not None]
losses  = [r["loss"]          for r in records if r.get("loss") is not None]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, linewidth=1.5, color="#1565C0", alpha=0.85)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss curve")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{WORKDIR}/training_curve.png", dpi=120)
plt.show()

total_drop = losses[0] - losses[-1]
print(f"Starting loss : {losses[0]:.4f}")
print(f"Final loss    : {losses[-1]:.4f}")
print(f"Total drop    : {total_drop:.4f}")
print()
if losses[-1] < 0.5:
    print("Loss is low - model has likely converged well.")
elif losses[-1] < 1.2:
    print("Loss is moderate - model has learned but may benefit from more epochs.")
else:
    print("Loss is still high - consider more epochs, a lower learning rate, or checking the data format.")

---
## 5. Merge the adapter and measure a baseline

Training kept the base weights frozen and only trained a small LoRA adapter. Merging folds
those deltas into the base weights, giving one standalone model with no adapter overhead.

The baseline below is the *same* base model, on the *same* validation set, with **no**
fine-tuning — a constrained seven-letter multiple-choice prompt so it can't fail just because
it phrases a label slightly differently. We measure it now, before attaching the adapter, so
the base model doesn't need to be loaded twice.

In [ ]:
import gc
import os
from pathlib import Path

import pandas as pd
import safetensors.torch as st
import torch
from peft import PeftConfig, get_peft_model
from peft.utils import set_peft_model_state_dict
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── USER INPUT ───────────────────────────────────────────────────────────────
# LABEL_TOKENS / LABEL_DISPLAY / display_labels / VAL_CSV come from cell 7.
CHOICES      = "ABCDEFG"
CHOICE2LABEL = {ch: lbl for ch, lbl in zip(CHOICES, LABEL_TOKENS)}

BASE_SYSTEM_PROMPT = (
    "You are a family-calendar coordinator's routing assistant. "
    "Classify the parent request by responding with ONLY a single letter - nothing else:\n"
    + "\n".join(f"{ch}) {lbl}" for ch, lbl in CHOICE2LABEL.items())
)

# ── Validate adapter ─────────────────────────────────────────────────────────
adapter_path = Path(ADAPTER_DIR)
if not adapter_path.is_dir():
    raise FileNotFoundError(f"Adapter folder not found: {ADAPTER_DIR!r}")
if not (adapter_path / "adapter_config.json").exists():
    raise FileNotFoundError(f"No adapter_config.json in {ADAPTER_DIR!r}")
print(f"Adapter found: {ADAPTER_DIR}")

# ── Device ───────────────────────────────────────────────────────────────────
HAS_CUDA = torch.cuda.is_available()
HAS_MPS  = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE   = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
dtype    = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
print(f"Device: {DEVICE}")

# ── Load base model + tokenizer ─────────────────────────────────────────────
print("\nLoading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=dtype, device_map=DEVICE,
)
tok_src    = str(adapter_path) if (adapter_path / "tokenizer.json").exists() else BASE_MODEL_NAME
_tokenizer = AutoTokenizer.from_pretrained(tok_src)
base_model.eval()

# ── Baseline inference on val split ─────────────────────────────────────────
_choice_ids = []
for ch in CHOICES:
    ids_plain  = _tokenizer.encode(ch,       add_special_tokens=False)
    ids_spaced = _tokenizer.encode(f" {ch}", add_special_tokens=False)
    _choice_ids.append(ids_spaced[0] if len(ids_spaced) == 1 else ids_plain[0])


def classify_base(request_text: str) -> str:
    messages = [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Parent request: {request_text}"},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model(input_ids=inputs["input_ids"])
    first_logits = out.logits[0, -1, :]
    choice_probs = torch.softmax(first_logits[_choice_ids], dim=-1).cpu().tolist()
    return CHOICE2LABEL[CHOICES[choice_probs.index(max(choice_probs))]]


df_val      = pd.read_csv(VAL_CSV)
y_true      = df_val["label"].tolist()
y_pred_base = []

for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Baseline inference"):
    y_pred_base.append(classify_base(row["text"]))

from sklearn.metrics import classification_report
print("\n=== Baseline (no fine-tuning) ===")
# labels=LABEL_TOKENS is required here: without it, sklearn silently sorts
# the true class strings alphabetically and zips target_names to THAT order
# instead of LABEL_TOKENS's order, printing every row under the wrong name
# (right numbers, wrong label) while looking completely normal.
print(classification_report(y_true, y_pred_base, labels=LABEL_TOKENS, target_names=display_labels, digits=3, zero_division=0))

# ── Attach LoRA + load weights ───────────────────────────────────────────────
print("Attaching LoRA adapter...")
peft_cfg   = PeftConfig.from_pretrained(str(adapter_path))
peft_model = get_peft_model(base_model, peft_cfg)

candidates = [
    adapter_path / "adapter_model.safetensors",
    adapter_path / "adapters.safetensors",
    adapter_path / "adapter_model.bin",
]
weights_file = next((p for p in candidates if p.exists()), None)
if weights_file is None:
    raise FileNotFoundError(f"No adapter weights in {ADAPTER_DIR}")

if weights_file.suffix == ".safetensors":
    adapter_weights = st.load_file(str(weights_file), device=DEVICE)
else:
    adapter_weights = torch.load(str(weights_file), map_location=DEVICE)

set_peft_model_state_dict(peft_model, adapter_weights)

# ── Merge + save to disk ─────────────────────────────────────────────────────
print("Merging (this may take a while) ...")
_model = peft_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
_model.save_pretrained(MERGED_DIR)
_tokenizer.save_pretrained(MERGED_DIR)
stale = Path(MERGED_DIR) / "adapter_config.json"
if stale.exists():
    stale.unlink()
print(f"Saved -> {MERGED_DIR}")

del peft_model, base_model, adapter_weights
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_model.eval()
print("Model ready for inference.")

---
## 6. `classify()` and a smoke test

Chat-format the request with the same system prompt used in training, generate a few tokens,
and match the start of the output to one of the seven labels. `classify()` also reports a
confidence score — the softmax probability of the winning label's first token versus the
others. Seven obvious requests, one per class, should all route correctly before trusting
the full validation run.

In [ ]:
import torch

# Must exactly match the SYSTEM_PROMPT used to build the training data in
# cell 7 — this is the v3, 7-label prompt (fast_path_conflict split out of
# fast_path_reject). A mismatch here would train on one framing and query
# with another, invalidating the comparison to earlier passes.
SYSTEM_PROMPT = (
    "You are a family-calendar coordinator's routing assistant. Given a parent's "
    "request, respond with exactly one of the following seven categories:\n"
    "- fast_path_mutate: a single, specific create/update/move/delete/restore of "
    "one event or reminder, pending approval. Use this for ordinary cancel/change/"
    "add requests with no past-dated event, no scheduling conflict, and no "
    "cross-family issue.\n"
    "- fast_path_read: a single, specific read-only list/show request for events "
    "already on the calendar. No mutation.\n"
    "- fast_path_reject: use ONLY when the request itself states an explicit "
    "disqualifying fact that is NOT a scheduling conflict — a clearly past-dated "
    "event, cross-family access (the named child/parent does not belong to the "
    "stated family), a stale plan version, or a repeat of something already "
    "rejected. Do not use this as a default when merely uncertain.\n"
    "- fast_path_conflict: use when the request would double-book the SAME child "
    "or parent into two overlapping activities or pickups. This is different from "
    "fast_path_reject — a conflict can be offered alternatives, not just declined.\n"
    "- deep_weekly_workflow: a broad request spanning multiple days or a whole "
    "week, not one event.\n"
    "- outing_workflow: a request to search for or suggest NEW outing/activity "
    "ideas — not to list or show events already on the calendar.\n"
    "- ambiguous_clarify: use ONLY when a specific required detail is missing "
    "(which event, which family, which time) and no other label clearly fits. Do "
    "not use this as a default when merely uncertain either.\n"
    "Respond with exactly one label and nothing else."
)
# LABEL_TOKENS / LABEL_DISPLAY / display_labels come from cell 7.


def classify(request_text: str, compute_confidence: bool = True) -> tuple:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Parent request: {request_text}"},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(_model.device)

    with torch.no_grad():
        out = _model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=_tokenizer.eos_token_id,
        )

    generated = _tokenizer.decode(
        out.sequences[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

    matched = next((l for l in LABEL_TOKENS if generated.lower().startswith(l.lower())), None)
    if matched is None:
        matched = "ambiguous_clarify"

    if not compute_confidence or not out.scores:
        return matched, 1.0

    first_scores = out.scores[0][0]
    probs = torch.softmax(first_scores, dim=-1)
    confidence = probs.max().item()
    return matched, confidence


SMOKE_TESTS = [
    ("Add Maya's swim class tomorrow from 4 to 5 PM for family-1", "fast_path_mutate"),
    ("Show today's activities for family-2", "fast_path_read"),
    ("Add Leo's soccer practice yesterday from 4 to 5 PM for family-1", "fast_path_reject"),
    ("Add Kai's chess club at 4pm Tuesday for family-1, but he already has tutoring session then", "fast_path_conflict"),
    ("Coordinate next week for family-1, resolve conflicts, and propose parent assignments", "deep_weekly_workflow"),
    ("Find three outdoor activities near San Jose for family-1 this weekend", "outing_workflow"),
    ("Delete soccer practice for family-1", "ambiguous_clarify"),
]

correct = 0
for text, expected in SMOKE_TESTS:
    pred, conf = classify(text)
    ok = "OK" if pred == expected else "MISMATCH"
    correct += pred == expected
    print(f"[{ok:8s}] expected={expected:22s} predicted={pred:22s} conf={conf:.1%}  | {text}")

print(f"\n{correct}/{len(SMOKE_TESTS)} smoke tests passed.")

---
## 7. Evaluate on the held-out validation split

These rows were never seen during training, so this is the honest read on routing accuracy.

In [ ]:
from tqdm.auto import tqdm

y_pred = []
for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Fine-tuned inference"):
    pred, _ = classify(row["text"], compute_confidence=False)
    y_pred.append(pred)

print(f"Evaluated {len(y_pred):,} samples.")

In [ ]:
from sklearn.metrics import classification_report

# labels=LABEL_TOKENS is required: without it, sklearn silently sorts the
# true class strings alphabetically and zips target_names to THAT order
# instead of LABEL_TOKENS's order, printing every row under the wrong name.
print(classification_report(y_true, y_pred, labels=LABEL_TOKENS, target_names=display_labels, digits=3))

print()
for i in [0, 5, 10, 15]:
    if i >= len(df_val):
        continue
    row  = df_val.iloc[i]
    pred, conf = classify(row["text"])
    print(f"=== Request {i}")
    print(f"TRUE label: {row['label']}")
    print(f"PRED label: {pred}  (conf {conf:.1%})")
    print(f"Text: {row['text']}")
    print()

### Reading the report

`fast_path_reject` and `fast_path_mutate` recall matter most here: a missed rejection means an
invalid mutation (a past event, cross-family access) slips further into the pipeline than it
should before the deterministic safeguards in `repository.py` catch it. `fast_path_conflict`
recall matters too, in a different way: a missed conflict (routed elsewhere instead) means a
double-booking goes through without ever getting the "offer alternatives / flag possible
error" treatment. A missed `deep_weekly_workflow` just costs a slower, single-turn answer to
what should have been a weekly plan — annoying, not unsafe.

### Confusion matrix

Rows are true labels, columns are predictions; the diagonal is correct. Colour is
row-normalised so it doubles as a recall heatmap. Watch for `fast_path_reject` rows leaking
into `fast_path_mutate` or `fast_path_conflict` columns — those are the confusion pairs worth
the most attention.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y_true, y_pred, labels=LABEL_TOKENS)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm, annot=cm, fmt="d", cmap="Blues",
    xticklabels=display_labels, yticklabels=display_labels,
    linewidths=0.5, ax=ax,
)
ax.set_title("Confusion matrix (counts shown, colour = row-normalised recall)")
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{WORKDIR}/confusion_matrix.png", dpi=120)
plt.show()

reject_recall = cm_norm[LABEL_TOKENS.index("fast_path_reject"), LABEL_TOKENS.index("fast_path_reject")]
print(f"\nfast_path_reject recall: {reject_recall:.1%}")
if reject_recall < 0.85:
    print("Warning: reject recall below 0.85 - consider more training data for this class.")

conflict_recall = cm_norm[LABEL_TOKENS.index("fast_path_conflict"), LABEL_TOKENS.index("fast_path_conflict")]
print(f"fast_path_conflict recall: {conflict_recall:.1%}")
if conflict_recall < 0.85:
    print("Warning: conflict recall below 0.85 - consider more training data for this class.")

---
## 8. Baseline vs fine-tuned

The baseline is the identical base model, same validation requests, zero task-specific
training — its best shot via the constrained seven-letter prompt. The gap between it and the
fine-tuned model is the measurable value of this labelled dataset and this training run.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report


def per_class_f1(y_t, y_p, labels):
    # labels=labels (sklearn's own ordering param) is required: without it,
    # sklearn silently sorts the true class strings alphabetically and zips
    # target_names to THAT order instead of `labels`'s order, so report[lbl]
    # below would silently fetch another class's f1-score.
    report = classification_report(y_t, y_p, labels=labels, target_names=labels, output_dict=True, zero_division=0)
    return {lbl: report[lbl]["f1-score"] for lbl in labels}


ft_f1    = per_class_f1(y_true, y_pred,      LABEL_TOKENS)
base_f1  = per_class_f1(y_true, y_pred_base, LABEL_TOKENS)
ft_acc   = accuracy_score(y_true, y_pred)
base_acc = accuracy_score(y_true, y_pred_base)

labels_plot = display_labels + ["overall accuracy"]
ft_vals     = [ft_f1[l]   for l in LABEL_TOKENS] + [ft_acc]
base_vals   = [base_f1[l] for l in LABEL_TOKENS] + [base_acc]

x     = np.arange(len(labels_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_base = ax.bar(x - width/2, base_vals, width, label="Base model (no fine-tuning)", color="#90CAF9", edgecolor="white")
bars_ft   = ax.bar(x + width/2, ft_vals,   width, label="Fine-tuned (LLaMA Board LoRA)", color="#1565C0", edgecolor="white")

ax.bar_label(bars_base, fmt="{:.2f}", padding=3, fontsize=8)
ax.bar_label(bars_ft,   fmt="{:.2f}", padding=3, fontsize=8)
bars_base[-1].set_color("#FFCC80")
bars_ft[-1].set_color("#E65100")

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels([l.replace(" ", "\n") for l in labels_plot], fontsize=9)
ax.set_ylabel("F1 score / Accuracy")
ax.set_title("Family request router - baseline vs fine-tuned")
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.15), ncol=2)
plt.tight_layout()
plt.savefig(f"{WORKDIR}/baseline_vs_finetuned.png", dpi=120)
plt.show()

print(f"Baseline accuracy    : {base_acc:.1%}")
print(f"Fine-tuned accuracy  : {ft_acc:.1%}")
print(f"Delta                : {(ft_acc - base_acc) * 100:+.1f} pts")

## Recap

- Fine-tuned `Qwen/Qwen3-1.7B-Base` with a LoRA adapter through LLaMA Board
  on `data/family_request_routing.csv` — 30 real, eval-labelled parent requests (sourced from
  [family-calendar](https://github.com/anushaakkiraju26/family-calendar)'s evaluation suite)
  expanded via `tools/generate_dataset.py` plus several hand-written contrastive expansions.
- Compared the merged fine-tuned model against a constrained-choice baseline on a held-out
  validation split, across seven training passes that each isolate one variable — prompt,
  data, label structure, or hyperparameters.

### Bug found: `classification_report` tables and the F1 chart were mislabeled

Every printed classification report and the baseline-vs-fine-tuned bar chart, across every
pass including before any of this notebook's edits, called `classification_report(...,
target_names=...)` without also passing `labels=`. Without it, `sklearn` silently sorts the
true class strings **alphabetically** and zips `target_names` to that order — not the order
`LABEL_TOKENS`/`display_labels` were built in. The numbers were always real; they were just
printed under the wrong row's name, in a way that looked completely normal. The confusion
matrix and the `reject_recall`/`conflict_recall` checks were never affected (they already
passed `labels=LABEL_TOKENS` explicitly) — which is exactly why they kept silently disagreeing
with the report table on every pass, and why this took so long to catch. Every per-class
recall in this Recap was cross-checked against confusion matrix images before being recorded
here, so those numbers hold up. Fixed everywhere `classification_report` is called (Sections
5, 7, 8) by passing `labels=LABEL_TOKENS`; re-verified fixed in pass 5's run (report and
confusion matrix now agree exactly).

### Results — passes 1-3 (6-label taxonomy)

| Metric | Baseline | Pass 1 (minimal prompt) | Pass 2 (richer prompt, same data) | Pass 3 (richer prompt, expanded data) |
|---|---|---|---|---|
| Overall accuracy | ~24.5% | 36.9% (+12.3 pts) | 46.2% (+21.5 pts) | **58.1%** (+33.8 pts) |

Per-class recall:

| Label | Pass 1 | Pass 2 | Pass 3 | Pass 2 -> 3 |
|---|---|---|---|---|
| `fast_path_read` | 100% | 100% | 100% | — |
| `deep_weekly_workflow` | 0% | 10% | **58.3%** | **+48.3 pts** |
| `ambiguous_clarify` | 0% | 0% | **35.7%** | **+35.7 pts** |
| `fast_path_mutate` | 63.6% | 54.5% | 66.7% | +12.2 pts |
| `outing_workflow` | 18.2% | 72.7% | 72.7% | — |
| `fast_path_reject` | 36.4% | 36.4% | 28.6% | −7.8 pts |

(Pass 1/2 used the original 65-row val split; pass 3 used the new 74-row split after the
dataset grew to 369 rows — per-class recall is still comparable as a rate, but not the same
exact validation examples.)

### What each pass changed, and what it fixed

**Pass 1 -> 2** changed only the training/inference `SYSTEM_PROMPT` (same 321-row dataset) —
added per-label definitions and explicit "don't use this as a default when uncertain"
guidance. This **fully resolved** the `outing_workflow` <-> `fast_path_read` confusion
(18.2% -> 72.7% recall) but had **no effect at all** on the `fast_path_reject` /
`ambiguous_clarify` / `deep_weekly_workflow` cluster — `ambiguous_clarify` stayed at exactly
0% recall in both passes. Conclusion: prompt instructions fix confusions that are
*lexical* (which surface cue to attend to) but not confusions that require the model to
*reason* about the request, when the training data itself doesn't teach that reasoning.

**Pass 2 -> 3** kept the pass-2 prompt fixed and changed only the data: fixed 4
`fast_path_reject` rows whose disqualifying fact (cross-family membership) was never actually
stated in the text — those rows were literally unsolvable from the text alone, and were
textually near-identical to several `ambiguous_clarify` rows, which is a likely contributor to
`ambiguous_clarify`'s 0% recall in pass 2. Added 48 hand-written contrastive examples for
`fast_path_reject`, `ambiguous_clarify`, `deep_weekly_workflow`, and `fast_path_mutate`. This
**broke the reject-as-catch-all collapse**: `deep_weekly_workflow` and `ambiguous_clarify` both
went from near-zero to real performance (+48.3 and +35.7 pts respectively).

Pass 3's remaining weak spot was `fast_path_reject` <-> `ambiguous_clarify`: no longer a
one-sided collapse, but still genuinely confused, roughly symmetrically (6/14 true-reject val
examples leaking into ambiguous_clarify, 8/14 true-ambiguous leaking into reject). Both
smoke-test failures in every pass were this exact pair.

### Restructure: `fast_path_conflict` split out of `fast_path_reject`

`fast_path_reject` was conflating two different things — an outright, deterministic decline
(past-dated event, cross-family access, stale plan, repeat-after-rejection) and a scheduling
conflict (the same child or parent double-booked). In the real Family Coordinator these need
**different downstream handling** — a conflict can be offered alternatives or flagged as a
likely parent error, not just declined — so this became a genuine 7th routing label,
`fast_path_conflict`: 16 existing `fast_path_reject` rows that were actually conflicts got
relabeled, 35 new hand-written `fast_path_conflict` examples were added, `fast_path_reject`'s
definition was narrowed to explicitly exclude conflicts, and the baseline's constrained-choice
grew from six letters to seven.

### Pass 4 results — 7-label taxonomy

| Metric | Baseline | Pass 4 |
|---|---|---|
| Overall accuracy | 28.4% | 55.6% (+27.2 pts) |

Per-class, fine-tuned:

| Label | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| `fast_path_read` | 0.688 | 1.000 | 0.815 | 11 |
| `fast_path_conflict` | 0.588 | **1.000** | 0.741 | 10 |
| `fast_path_mutate` | 0.353 | 1.000 | 0.522 | 12 |
| `deep_weekly_workflow` | 1.000 | 0.500 | 0.667 | 12 |
| `outing_workflow` | 1.000 | 0.364 | 0.533 | 11 |
| `fast_path_reject` | 0.500 | 0.182 | 0.267 | 11 |
| `ambiguous_clarify` | 0.000 | **0.000** | 0.000 | 14 |

`fast_path_conflict` was an immediate, clean success — 100% recall on its very first training
pass, validating the split as a real, learnable distinction. Everything downstream of the
split got harder, not easier: `fast_path_reject` dropped to its worst recall yet, and
`ambiguous_clarify`'s confusion **target moved** from `fast_path_reject` (passes 1-3) to
`fast_path_mutate` — the "catch-all default when uncertain" behavior isn't tied to a specific
label, it relocates to whichever label is the current path of least resistance.
`outing_workflow` also regressed unexpectedly (72.7% -> 36.4%), with no change to its own data
or prompt since pass 3.

### Pass 5: bugfix verification (no data/prompt/hyperparameter change)

Re-ran pass 4's exact same trained adapter through the now-fixed evaluation cells. Confirmed
byte-identical smoke-test and loss-curve output to pass 4 (same model), and confirmed the
classification report, confusion matrix, and bar chart now all agree with each other for the
first time. No new findings — this pass exists purely to validate the bugfix.

### Pass 6: minimal-pair contrastive data for reject/clarify/mutate

`ambiguous_clarify` had been stuck at or near 0% recall across every prior structural change.
Added 15 minimal-pair triplets (44 new rows, 448 total): each triplet holds a base request
constant and varies exactly one fact — a stated disqualifying fact (past date, cross-family,
retry-after-rejection, stale plan) -> `fast_path_reject`; that same fact **omitted entirely**
-> `ambiguous_clarify`; stated and fine (future date, correct family) -> `fast_path_mutate`.
Same prompt, same LoRA rank/epochs as every pass since pass 1 — only the data changed.

| Label | Pass 5 | Pass 6 | Change |
|---|---|---|---|
| `fast_path_mutate` | 100% | 100% | — |
| **`fast_path_reject`** | 18.2% | **42.9%** | **+24.7 pts** |
| `fast_path_read` | 100% | 81.8% | −18.2 pts |
| `fast_path_conflict` | 100% | 60.0% | −40 pts |
| `deep_weekly_workflow` | 50% | 33.3% | −16.7 pts |
| `outing_workflow` | 36.4% | 9.1% | −27.3 pts |
| **`ambiguous_clarify`** | 0% | **0%** | **unchanged** |
| **Overall accuracy** | 55.6% | **45.6%** | **−10 pts** |

**Two important findings, one encouraging and one not.** `fast_path_reject` improved
substantially (+24.7 pts) — the minimal-pair examples clearly helped it specifically. But
`ambiguous_clarify` **did not move at all**, even under the most targeted intervention tried
so far — a real negative result. Three different fixes (richer prompt, general contrastive
data, minimal-pair contrastive data) have now all failed to move this one class, which
pass 6 read as pointing away from "needs better/more data" and toward a structural
bottleneck: LoRA rank 8 possibly lacking the capacity to represent this specific three-way
boundary on top of everything else it's already carrying. Pass 7 below tests that hypothesis
directly.

Meanwhile several *other* classes got worse (`fast_path_conflict` −40pts, `outing_workflow`
−27.3pts, `deep_weekly_workflow` −16.7pts) even though their own data and the prompt didn't
change. `outing_workflow` in particular has now declined for three straight passes
(72.7% -> 36.4% -> 9.1%) with zero changes to its own boundary. The likely mechanism: the
dataset has grown unevenly (`fast_path_reject`/`fast_path_mutate`/`ambiguous_clarify` gained
the most rows across passes 3, 4, and 6) while LoRA rank stayed fixed at 8 — as more of the
adapter's limited capacity gets pulled toward the classes with the most training mass,
boundaries that were previously working (like outing/read, fixed back in pass 2) appear to be
getting crowded out rather than staying learned.

### Pass 7: LoRA rank 8 -> 16 (same 448-row dataset, same prompt, same 3 epochs)

Directly tested pass 6's capacity-bottleneck hypothesis: same data, same `SYSTEM_PROMPT`,
same 3 epochs, only the LoRA rank changed. This pass also survived a Colab runtime disconnect
mid-investigation, which is worth recording since it produced a confusing false alarm before
the real result: the first two re-runs after reconnecting silently retrained (or re-evaluated)
at the **old rank-8 default** — HF Trainer's fixed seed (42) means any run with identical
hyperparameters, rank included, reproduces bit-identical loss curves and predictions, so two
apparently-different "rank 16" attempts came back byte-for-byte matching pass 6 before the
rank change had actually taken effect.

Once a genuine rank-16 run was confirmed — not by trusting the UI or even `adapter_config.json`
alone, but by checking three independent things: (1) the merge cell's own printed
`Adapter found: .../train_2026-09-12-06-48-58` matched the intended run's folder, (2) that
folder's `adapter_config.json` recorded `"r": 16`, and (3) the actual saved tensor
`lora_A.weight` had shape `[16, 6144]` (double rank 8's width) — the result was:

| Metric | Pass 6 (rank 8) | Pass 7 (rank 16) | Change |
|---|---|---|---|
| Overall accuracy | 45.6% | **45.6%** | **none** |
| `fast_path_mutate` | 100% | 100% | — |
| `fast_path_read` | 81.8% | 81.8% | — |
| `fast_path_reject` | 42.9% | 42.9% | — |
| `fast_path_conflict` | 60.0% | 60.0% | — |
| `deep_weekly_workflow` | 33.3% | 33.3% | — |
| `outing_workflow` | 9.1% | 9.1% | — |
| `ambiguous_clarify` | 0% | 0% | — |
| Training loss (start -> final) | 4.8606 -> 0.3045 | 4.8606 -> 0.3045 | **identical** |
| Smoke tests | 5/7 | 5/7 (same two failures) | — |

**Doubling LoRA rank had zero measurable effect on anything**, reproduced across multiple
reruns once the genuine rank-16 adapter was confirmed. This is a real negative result, not
another instance of the stale-adapter mixup: the weights really are rank 16, the merge really
loaded them, and the outcome is indistinguishable from rank 8 down to the training loss curve.

**This rules out pass 6's capacity hypothesis.** If LoRA rank were the bottleneck, doubling it
should have moved *something* — even a small shift in `ambiguous_clarify`'s recall or in one
of the classes that regressed in pass 6. Getting an outcome this identical instead points to
the model already representing everything it's going to learn from this data at rank 8, with
more capacity simply going unused. `ambiguous_clarify` has now failed to move under four
different interventions across two label taxonomies (richer prompt, general contrastive data,
minimal-pair contrastive data, doubled LoRA rank) — the remaining live hypotheses are the
data itself (its examples may be too textually close to `fast_path_mutate`'s, which is where
100% of its errors land in pass 6/7) or an untested hyperparameter other than rank, most
notably epoch count (still 3 throughout every pass since pass 1).

### Where this could go next

1. ~~Test LoRA rank as the next single-variable change~~ — **done in pass 7**: rank 8 -> 16
   produced a bit-identical result, ruling out adapter capacity as the bottleneck. Not worth
   revisiting without changing something else alongside it.
2. **Audit `ambiguous_clarify`'s examples directly against `fast_path_mutate`'s**, side by
   side — since neither more data, more contrastive structure, nor more model capacity has
   moved this boundary at all, the remaining hypothesis is that the two classes' example texts
   or definitions genuinely overlap in a way none of these interventions can fix without a
   difference in the training signal itself.
3. **Try more epochs** (still 3 throughout all 7 passes) as the other untested single-variable
   change, distinct from rank — the loss curve in every pass has still been dropping steeply
   at step 35, so more epochs may extract more from the same rank-8 adapter than a wider one
   did.
4. **Production integration**: wire the merged model in as a cheap pre-filter ahead of the
   Family Coordinator's own routing reasoning in the family-calendar project — call it first,
   and only fall back to the full frontier-model reasoning path when its confidence is low or
   the request lands in `ambiguous_clarify`. Not required for this submission, but it's the
   actual production version of the "small model as a routing gate" idea the reference project
   argues for.